# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/usmanCh129/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# 1. My lane: Refresh / Content Opportunity Scoring

I am choosing **Refresh / Content Opportunity Scoring** as my provisional lane. I want to investigate which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring using observable search-performance and content signals.

I chose this lane because the starter dataset provides page-level visibility, click, position, engagement, freshness, and content signals, and my Week 1–2 experiments showed that these signals can support useful ranking decisions.

I will keep this lane provisional and refine the exact target and action definition as I learn more from the full warehouse and future time windows.


In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# 2. The question: decision, action, cost of a wrong call

## Research question

**Which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring, based on observable search-performance and content signals?**

### Decision

The decision is **which pages should receive limited SEO/content-review time first**.

### Unit of analysis

The unit of analysis is **one content page**.

### Output

The intended output is a **ranked review queue**. Each page should eventually have a priority score, a suggested action, understandable reason codes, and a confidence indication.

### Who acts on it?

An SEO specialist or content editor would use the queue to decide which pages to inspect first.

### Action

The reviewer could investigate a high-priority page and decide whether it needs a refresh, expansion, protection, pruning, consolidation investigation, or simply monitoring.

The system will support the decision rather than automatically deciding that a page must be changed.

### Cost of a wrong recommendation

A false positive can waste limited editorial time on a page that does not need attention. A false negative can cause a potentially valuable opportunity or declining page to be missed.

Because review capacity is limited, the quality of the highest-ranked pages matters. I therefore expect **Precision@K** to be an important success metric for the ranking task.

### Why data or ML can help

A simple rule can identify obvious cases such as stale pages with meaningful visibility. However, page performance may depend on several signals at once, including impressions, clicks, CTR, position, freshness, content age, word count, and engagement.

I will first establish a transparent baseline. ML earns its place only if combining these signals produces a more useful ranking than a simple rule, and if the improvement survives leakage-safe validation.
## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# 3. Quick look at the data

I will use the starter dataset to check whether the Refresh / Content Opportunity lane is worth investigating further.

The code below calculates the dataset size, the relationship between search volume and actual impressions, and the number of pages receiving meaningful impressions but zero clicks.

### What these numbers suggest

The starter dataset contains 30,000 page-level records across 44 columns. The correlation between `search_volume` and `impressions_90d` is **0.001**, which is effectively zero in this dataset. This suggests that keyword search volume alone is not a useful linear proxy for the actual search exposure received by an individual page.

A second signal is that **6,068 pages (20.23%)** received at least 100 impressions but recorded zero clicks. These pages are not automatically problems; zero clicks could have several explanations. However, they represent a large enough group to justify investigation into whether observable page and search signals can help prioritize which pages deserve human review.

Together, these observations suggest that a useful content-opportunity system should not rely on a single metric. I will investigate whether multiple observable signals can produce a more useful and explainable review ranking than a simple hand-written rule.

The goal is therefore not to automatically decide what should happen to a page. The goal is to improve the **order in which a limited human review capacity is applied**.

In [28]:
import os
import sys
import subprocess

REPO_URL = "https://github.com/usmanCh129/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

# Clone your repository if it isn't already present
if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

# Move into your repository
os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())
print("Repository contents:")
print(os.listdir())

# Confirm the dataset exists
DATA_PATH = "data/raw/content_refresh_anonymized.csv"

if os.path.exists(DATA_PATH):
    print("✅ Dataset found:", DATA_PATH)
else:
    print("❌ Dataset not found:", DATA_PATH)

Working directory: /content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship
Repository contents:
['DATA_USE.md', 'README.md', 'notebooks', 'docs', 'work', 'SETUP.md', 'data', 'scripts', 'skills', 'outputs', '.git', 'AGENTS.md', 'LICENSE', '.github', 'submission', 'CLAUDE.md', 'GUIDE.md', '.gitignore', 'requirements.txt']
✅ Dataset found: data/raw/content_refresh_anonymized.csv


In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

corr = df["search_volume"].corr(df["impressions_90d"])
print(f"Search volume vs impressions correlation: {corr:.3f}")

zero_click = df[
    (df["impressions_90d"] >= 100) &
    (df["clicks_90d"] == 0)
]

print(
    f"Pages with >=100 impressions but 0 clicks: "
    f"{len(zero_click):,}"
)

print(
    f"Share of all pages: "
    f"{len(zero_click) / len(df) * 100:.2f}%"
)

Rows: 30,000
Columns: 44
Search volume vs impressions correlation: 0.001
Pages with >=100 impressions but 0 clicks: 6,068
Share of all pages: 20.23%


# 4. Careful words: what I can and can't claim

## What I can claim

I can identify and measure patterns observed in the anonymized starter dataset.

I can compare simple ranking rules with ML-based ranking approaches and report whether one approach performs better under a defined evaluation setup.

I can produce a decision-support queue that helps a human reviewer prioritize pages for investigation.

For the eventual capstone, I can test whether signals observed before a decision point are useful for predicting a clearly defined future outcome, provided the feature and target windows are separated and the evaluation is leakage-safe.

## What I cannot claim

I cannot claim that the observed relationships prove causation.

I cannot claim that a page is guaranteed to recover because it is recommended for review.

I cannot claim that the system predicts Google's ranking algorithm.

I cannot assume that results from the 30,000-row starter dataset automatically generalize to the full warehouse.

I also will not use `trend_direction` or `trend_pct` as model features when predicting the starter decline label, because the label is derived from those fields and using them would create leakage.

My conclusions will therefore use careful terms such as **observed, measured, directional, candidate, ranking, and decision-support**.


In [30]:

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.